## Loading the Data

In [ ]:
import os

In [ ]:
dataset_root = "/kaggle/input/private-dataset/Chest X-ray dataset for lung segmentation"

for root, dirs, files in os.walk(dataset_root):
    print(root, len(files))
    if len(files) > 0:
        print("  sample files:", files[:3])

In [ ]:
dataset_roots = [
    "/kaggle/input/private-dataset/Chest X-ray dataset for lung segmentation/Darwin/Darwin",
    "/kaggle/input/private-dataset/Chest X-ray dataset for lung segmentation/Montgomery/Montgomery",
    "/kaggle/input/private-dataset/Chest X-ray dataset for lung segmentation/Shenzhen/Shenzhen",
]

## Sanity Check

In [ ]:
import os

for root in dataset_roots:
    img_dir = os.path.join(root, "img")
    mask_dir = os.path.join(root, "mask")

    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    mask_files = set([f for f in os.listdir(mask_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])

    missing = [f for f in img_files if f not in mask_files]
    extra_masks = [f for f in mask_files if f not in set(img_files)]

    print(f"\nDataset: {root}")
    print("Images:", len(img_files))
    print("Masks:", len(mask_files))
    print("Missing masks:", len(missing))
    print("Extra masks:", len(extra_masks))

    if missing:
        print("Missing examples:", missing[:10])
    if extra_masks:
        print("Extra mask examples:", extra_masks[:10])

## Dataset Class

In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

class LungSegmentationDataset(Dataset):
    def __init__(self, dataset_roots, image_size=256):
        self.samples = []
        self.image_size = image_size

        for root in dataset_roots:
            img_dir = os.path.join(root, "img")
            mask_dir = os.path.join(root, "mask")

            if not os.path.isdir(img_dir):
                raise FileNotFoundError(f"Missing img folder: {img_dir}")
            if not os.path.isdir(mask_dir):
                raise FileNotFoundError(f"Missing mask folder: {mask_dir}")

            image_names = sorted([
                f for f in os.listdir(img_dir)
                if f.lower().endswith((".png", ".jpg", ".jpeg"))
            ])

            for image_name in image_names:
                image_path = os.path.join(img_dir, image_name)
                mask_path = os.path.join(mask_dir, image_name)

                if os.path.exists(mask_path):
                    self.samples.append((image_path, mask_path))
                else:
                    print(f"Warning: missing mask for {image_name}")

        if len(self.samples) == 0:
            raise ValueError("No image-mask pairs found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, mask_path = self.samples[idx]

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")
        if mask is None:
            raise FileNotFoundError(f"Could not read mask: {mask_path}")

        image = cv2.resize(image, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        image = image.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)

        image = np.expand_dims(image, axis=0)   # (1, H, W)
        mask = np.expand_dims(mask, axis=0)     # (1, H, W)

        image = torch.tensor(image, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.float32)

        return image, mask

In [ ]:
import matplotlib.pyplot as plt
import torch

dataset = LungSegmentationDataset(dataset_roots, image_size=256)

print("Total samples:", len(dataset))

image, mask = dataset[0]
print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Image min/max:", image.min().item(), image.max().item())
print("Mask unique values:", torch.unique(mask))

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(image.squeeze(0), cmap="gray")
plt.title("X-ray")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask.squeeze(0), cmap="gray")
plt.title("Mask")
plt.axis("off")

plt.tight_layout()
plt.show()

## Train/Val Split

In [ ]:
from torch.utils.data import random_split, DataLoader

full_dataset = LungSegmentationDataset(dataset_roots, image_size=256)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

## Define U-Net

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.final_conv = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final_conv(d1)

## Loss (BCE + Dice)

In [ ]:
import torch.nn.functional as F

def dice_loss(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice.mean()

def combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    d = dice_loss(logits, targets)
    return bce + d

## Training Setup

In [ ]:
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(in_channels=1, out_channels=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 12
best_val_loss = float("inf")

## Training Loop

In [ ]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} Train"):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = combined_loss(logits, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} Val"):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            logits = model(images)
            loss = combined_loss(logits, masks)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/kaggle/working/best_unet.pth")
        print("Saved best model.")

## Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt

model.load_state_dict(torch.load("/kaggle/input/datasets/junyongha/best-unet/best_unet.pth", map_location=device))
model.eval()

images, masks = next(iter(val_loader))
images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    logits = model(images)
    preds = (torch.sigmoid(logits) > 0.5).float()

images = images.cpu()
masks = masks.cpu()
preds = preds.cpu()

for i in range(min(3, len(images))):
    plt.figure(figsize=(10, 3))

    plt.subplot(1, 3, 1)
    plt.imshow(images[i].squeeze(0), cmap="gray")
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(masks[i].squeeze(0), cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(preds[i].squeeze(0), cmap="gray")
    plt.title("Prediction")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

## Build Dataset for Classification

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

data_dir = "/kaggle/input/datasets/danielmadmon/nih-chest-xray-13-labels/Classes"

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
# print("Classes:", full_dataset.classes)
# print("Total samples:", len(full_dataset))

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

## Build Classifier

In [ ]:
class LungClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.enc1 = DoubleConv(1, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0, 5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.pool1(self.enc1(x))
        x = self.pool2(self.enc2(x))
        x = self.pool3(self.enc3(x))
        return self.classifier(x)

## Training the Classifier without Unet Model and Masking

In [ ]:
classifier = LungClassifier(num_classes=13).to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
best_val_loss = float("inf")

for epoch in range(15):
    classifier.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = classifier(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    classifier.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Train"):
            images, labels = images.to(device), labels.to(device)
            outputs = classifier(images)
            val_loss += criterion(outputs, labels).item()
        val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(classifier.state_dict(), "/kaggle/working/best_classifier.pth")
        print("Saved best model.")

## Training the Classifier with Unet Model and Masking

In [ ]:
classifier = LungClassifier(num_classes=13).to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
best_val_loss = float("inf")

seg_model = UNet(in_channels=1, out_channels=1).to(device)
seg_model.load_state_dict(torch.load("/kaggle/working/best_unet.pth", map_location=device))
seg_model.eval()

def apply_mask(images):
    with torch.no_grad():
        masks = (torch.sigmoid(seg_model(images)) > 0.5).float()
    return images * masks

for epoch in range(15):
    classifier.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        images, labels = images.to(device), labels.to(device)
        images = apply_mask(images)
        optimizer.zero_grad()
        outputs = classifier(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    classifier.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} Train"):
            images, labels = images.to(device), labels.to(device)
            images = apply_mask(images)
            outputs = classifier(images)
            val_loss += criterion(outputs, labels).item()
        val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(classifier.state_dict(), "/kaggle/working/best_classifier.pth")
        print("Saved best model.")

## Metrics

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_model(model, loader, use_mask=False):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if use_mask:
                images = apply_mask(images)
            preds = model(images).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_preds, all_labels

# Without segmentation
classifier_no_seg = LungClassifier(num_classes=13).to(device)
classifier_no_seg.load_state_dict(torch.load("/kaggle/working/best_classifier_noseg.pth"))
preds_no_seg, labels = evaluate_model(classifier_no_seg, val_loader, use_mask=False)

# With segmentation
classifier_seg = LungClassifier(num_classes=13).to(device)
classifier_seg.load_state_dict(torch.load("/kaggle/working/best_classifier.pth"))
preds_seg, _ = evaluate_model(classifier_seg, val_loader, use_mask=True)

print("WITHOUT segmentation:")
print(f"  Accuracy: {accuracy_score(labels, preds_no_seg):.4f}")
print(f"  F1 Score: {f1_score(labels, preds_no_seg, average='macro'):.4f}")

print("\nWITH segmentation:")
print(f"  Accuracy: {accuracy_score(labels, preds_seg):.4f}")
print(f"  F1 Score: {f1_score(labels, preds_seg, average='macro'):.4f}")